**This script processes multiple prompts and data files in batch using Hugging Face transformer libraries**

1. Model Initialization: Loads any Hugging Face model once and reuses it for all processing

2. List Management:
Reads prompt filenames from prompt_list.txt
Reads data filenames from data_list.txt


3. Batch Processing:
Iterates through each prompt file
For each prompt, processes it with each data file
Generates responses for all prompt-data combinations

4. Outputs:
Saves outputs using the prompt filename as base
Adds timestamps to avoid overwriting
Organizes all outputs in an outputs/ directory

In [2]:
#pip install transformers

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from pathlib import Path
import json
from datetime import datetime

In [4]:
def initialize_model(model_name):
    # Initialize a Hugging Face model and tokenizer.

    print(f"Initializing model: {model_name}")
    print("-" * 50)

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )

    # Set padding token if not present
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"✓ Model '{model_name}' loaded successfully!")
    print("-" * 50)
    return model, tokenizer

In [5]:
def load_file_list(list_file):
    # Load a list of filenames from a text file.

    try:
        with open(list_file, 'r', encoding='windows-1252') as f:
            # Read lines and remove empty lines and whitespace
            filenames = [line.strip() for line in f.readlines() if line.strip()]
        print(f"✓ Loaded {len(filenames)} filenames from {list_file}")
        return filenames
    except FileNotFoundError:
        print(f"✗ Error: {list_file} not found!")
        return []

In [6]:
def load_prompt_file(prompt_file):
    # Load prompt content from a text file.

    try:
        with open(prompt_file, 'r', encoding='windows-1252') as f:
            prompt = f.read().strip()
        return prompt
    except FileNotFoundError:
        print(f"  ✗ Warning: Prompt file '{prompt_file}' not found. Using default prompt.")
        return "Please analyze the following data and provide insights:"

In [7]:
def load_data_file(data_file):
    # Load data from a text file.

    try:
        with open(data_file, 'r', encoding='windows-1252') as f:
            data = f.read().strip()
        return data
    except FileNotFoundError:
        print(f"  ✗ Warning: Data file '{data_file}' not found. Using empty data.")
        return ""

In [8]:
def generate_response(model, tokenizer, prompt, data, max_length):
    # Generate response from the model given prompt and data.

    # Combine prompt and data
    full_input = f"Prompt:\n{prompt}\n\nData:\n{data}\n\nResponse:"

    print(f"{full_input}")

    # Tokenize input
    inputs = tokenizer(
        full_input,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    # Move to same device as model
    if torch.cuda.is_available():
        inputs = inputs.to(model.device)

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.2,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the generated part (after "Response:")
    if "Response:" in response:
        response = response.split("Response:")[-1].strip()

    return response

In [9]:
def save_response(response, prompt_name, data_file, output_dir="outputs"):
    # Save the model response to a text file using the prompt name.

    # Create output directory if it doesn't exist
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Generate output filename based on prompt name
    base_name = Path(prompt_name).stem  # Remove extension
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = Path(output_dir) / f"{base_name}_response_{timestamp}.txt"

    # Save response
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"Prompt File: {prompt_name}\n")
        f.write(f"Data File: {data_file}\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("-" * 50 + "\n\n")
        f.write(response)

    print(f"  ✓ Response saved to: {output_file}")
    return output_file

In [10]:
def process_batch(model, tokenizer, prompt_files, data_files, max_length):
    # Process all combinations of prompts and data files.

    output_files = []
    num_pairs = min(len(prompt_files), len(data_files))
    current = 0

    print(f"\nProcessing prompt pairs...")
    print("=" * 50)

    # Process this single pair
    for i in range(num_pairs):
        prompt_file = prompt_files[i]   # Get prompt at index i
        data_file = data_files[i]       # Get data at same index i

        # Load prompt once for all data files
        prompt_text = load_prompt_file(prompt_file)

        # Load data
        data_text = load_data_file(data_file)

        #print(f"\nPrompt: {prompt_file}")
        #print(f"\nData: {data_file}")

        try:
            # Generate response
            print(f"    Generating response...")
            response = generate_response(model, tokenizer, prompt_text, data_text, max_length)

            # Save response with prompt name
            output_file = save_response(response, prompt_file, data_file)
            output_files.append(output_file)

        except Exception as e:
            print(f"    ✗ Error processing combination: {str(e)}")
            continue

    return output_files

In [11]:
def create_summary_report(output_files, report_file="processing_summary.txt"):
    # Create a summary report of all processed files.

    with open(report_file, 'w', encoding='windows-1252') as f:
        f.write("BATCH PROCESSING SUMMARY\n")
        f.write("=" * 50 + "\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total files processed: {len(output_files)}\n\n")

        f.write("Output Files:\n")
        f.write("-" * 50 + "\n")
        for i, output_file in enumerate(output_files, 1):
            f.write(f"{i}. {output_file}\n")

    print(f"\n✓ Summary report saved to: {report_file}")

In [ ]:
# Main function

MODEL_NAME="Qwen/Qwen3-4B-Instruct-2507"

PROMPT_LIST_FILE="prompt_list.txt"
DATA_LIST_FILE="data_list.txt"
OUTPUT_DIR="outputs"
MAX_LENGTH=2048  # Maximum tokens to generate

print("\n" + "=" * 50)
print("BATCH PROCESSING PIPELINE")
print("=" * 50)

try:
    # Step 1: Initialize model
    model, tokenizer = initialize_model(MODEL_NAME)

    # Step 2: Load lists of prompt and data filenames
    prompt_files = load_file_list(PROMPT_LIST_FILE)
    data_files = load_file_list(DATA_LIST_FILE)

    if not prompt_files:
        print(f"\n✗ No prompt files found in {PROMPT_LIST_FILE}")
        print("Please create this file with one prompt filename per line.")
        #return

    if not data_files:
        print(f"\n✗ No data files found in {DATA_LIST_FILE}")
        print("Please create this file with one data filename per line.")
        #return

    print(f"\nFound {len(prompt_files)} prompts and {len(data_files)} data files")

    # Step 3: Process all combinations
    output_files = process_batch(
        model, tokenizer,
        prompt_files, data_files,
        max_length=MAX_LENGTH
    )

    # Step 4: Create summary report
    if output_files:
        create_summary_report(output_files)
        print("\n" + "=" * 50)
        print(f"✓ PROCESSING COMPLETE!")
        print(f"✓ Generated {len(output_files)} response files in '{OUTPUT_DIR}' directory")
    else:
        print("\n✗ No files were successfully processed.")

except Exception as e:
    print(f"\n✗ Critical error: {str(e)}")
    import traceback
    traceback.print_exc()


BATCH PROCESSING PIPELINE
Initializing model: Qwen/Qwen3-4B-Instruct-2507
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

... (6 outputs left out)

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✓ Model 'Qwen/Qwen3-4B-Instruct-2507' loaded successfully!
--------------------------------------------------
✓ Loaded 1 filenames from prompt_list.txt
✓ Loaded 1 filenames from data_list.txt

Found 1 prompts and 1 data files

Processing prompt pairs...
    Generating response...
Prompt:
Using the Data below. You are an urban planner and you’re writing a rezoning assessment report. For this section of the report, you must create a table with 3 columns called ‘SEPPs’. ‘Consistent / Not Applicable’ and ‘Reasons for Consistency or Inconsistency’. Insert the name of the SEPP in the first column in the table, insert “consistent” in the second column in the table. In the third column, insert the aim of the SEPP and discuss why it applies and address consistency or inconsistency. This section must start with the text “The planning proposal is consistent with all relevant SEPPs as discussed in the table below:” before the table.

Data:
Data
SEPP (Biodiversity and Conservation) 2021 Applicable.